# Eichenbaum regular price
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
regular_window = 7
reference_window = 7
case = "Comparison"

Load data:

In [ ]:
# load retailer data
data = pd.read_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window))

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

### Eichenbum regular price

In [ ]:
# Group by, then calculate mode for each group
def mode_price(group):
    return group.mode()[0] if not group.mode().empty else None

# Regular price function
def calculate_regular_price_eichenbaum(df, period = 7):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'window' column for grouping
    if period == 7:
        df['window'] = df['fecha'].dt.to_period('W-MON')
    elif period == 30:
        df['window'] = df['fecha'].dt.to_period('M')

    # Apply the mode calculation for each product and month
    regular_prices = df.groupby(['tienda', 'descripcion', 'window'])['precio'].apply(mode_price).reset_index()
    
    # Merge regular price back into the original dataframe
    df = df.merge(regular_prices, on=['tienda', 'descripcion', 'window'], how='left', suffixes=('', '_regular'))
    
    # Rename the new column for clarity
    df.rename(columns={'precio_regular': 'precio_re'}, inplace=True)
    
    # Drop the 'window' column as it's no longer needed
    df.drop(columns='window', inplace=True)
    
    return df

In [ ]:
eidf = calculate_regular_price_eichenbaum(data, period = reference_window)

In [ ]:
eidf.to_csv(wd_dpr + "Data_Regular_Price_{}_{}_{}.csv".format(case, regular_window, reference_window), index = False)